In [ ]:
"""
LIMPIEZA Y PREPROCESAMIENTO DE LOS DATOS
================================================================================
Fase 3 de CRISP-DM: Preparación de los Datos.

Aplica, en un solo bloque, las decisiones documentadas en el análisis
exploratorio:

  1. Eliminación de columnas no predictivas (rutas de imagen).
  2. Tratamiento de valores faltantes (eliminación e imputación).
  3. Eliminación de características redundantes por relaciones elevadas
     y dependencias matemáticas.
  4. Eliminación de orig_rango por su concentración marcada en los extremos.
  5. Codificación One-Hot de las variables categóricas nominales.
  6. Conservación de formato_entrada como información disponible antes
     de la recodificación.
  7. Conservación de LPIPS en su escala original.

Entrada:
    csv/dataset_final_orig.csv

Salida:
    csv/dataset_limpio.csv

El dataset de entrada contiene 29.991 filas y 33 columnas.
La salida se utilizará posteriormente para la separación de los datos
y el escalado de las variables durante la etapa de preparación para
el modelado.
"""

import pandas as pd
import numpy as np

CSV_PATH = "csv/dataset_final_orig.csv"  # <-- ajustar a la ruta real
df = pd.read_csv(CSV_PATH)
n_inicial = len(df)
print(f"Dataset cargado: {n_inicial} filas, {df.shape[1]} columnas")

# ------------------------------------------------------------------
# 1. ELIMINAR COLUMNAS NO PREDICTIVAS (rutas de archivo)
# ------------------------------------------------------------------
df = df.drop(columns=["img_original", "img_recodificada"])

# ------------------------------------------------------------------
# 2. TRATAMIENTO DE VALORES FALTANTES
# ------------------------------------------------------------------
# 2a. lpips: variable objetivo, no imputable -> eliminar filas
n_lpips_nulo = df["lpips"].isna().sum()
df = df.dropna(subset=["lpips"])

# 2b. orig_entropia_espectral: coincide exactamente con los 4 registros
#     degenerados (orig_hf = orig_lf = orig_hlfr = 0) -> eliminar filas
n_entesp_nulo = df["orig_entropia_espectral"].isna().sum()
df = df.dropna(subset=["orig_entropia_espectral"])

# 2c. orig_hue_var: un NaN no equivale a varianza circular = 0, sino a
#     ausencia de medicion valida (sin pixeles que superen el umbral de
#     saturacion). Imputar con 0 confundiria ambas condiciones. Se imputa
#     con la mediana de los casos validos (valor neutro, no interpretable
#     como una medicion real) y se agrega una columna indicadora binaria
#     que preserva la distincion entre "medido" y "no medible",
#     permitiendo que el modelo aprenda de la ausencia si resulta
#     informativa.
n_hue_nulo = df["orig_hue_var"].isna().sum()
mediana_hue = df["orig_hue_var"].median()
df["orig_hue_var_faltante"] = df["orig_hue_var"].isna().astype(int)
df["orig_hue_var"] = df["orig_hue_var"].fillna(mediana_hue)

print(f"\nFilas eliminadas por lpips nulo: {n_lpips_nulo}")
print(f"Filas eliminadas por orig_entropia_espectral nulo: {n_entesp_nulo}")
print(f"Valores de orig_hue_var imputados con la mediana ({mediana_hue:.5f}): {n_hue_nulo}")
print("Se agrego la columna 'orig_hue_var_faltante' como indicador binario")
print(f"Filas restantes: {len(df)}")
assert df.isna().sum().sum() == 0, "Quedan valores nulos sin tratar"

# ------------------------------------------------------------------
# 3. ELIMINAR CARACTERISTICAS REDUNDANTES POR MULTICOLINEALIDAD (|r|>0.9)
# ------------------------------------------------------------------
columnas_redundantes = [
    "orig_var_lum",   # redundante con orig_std (r=0.9741)
    "orig_edges_s1",  # redundante con orig_grad_mean (r=0.9004)
    "orig_edges_s2",  # redundante con orig_edges_s3 y orig_grad_mean
    "orig_hlfr",      # derivada deterministicamente de orig_hf/orig_lf
]
df = df.drop(columns=columnas_redundantes)

# ------------------------------------------------------------------
# 4. ELIMINAR orig_rango (comportamiento casi binario por construccion)
# ------------------------------------------------------------------
df = df.drop(columns=["orig_rango"])

# ------------------------------------------------------------------
# 5. CODIFICACION ONE-HOT DE CATEGORICAS NOMINALES
#    formato_entrada se conserva: es informacion conocida antes de la
#    recodificacion y potencialmente predictiva (p. ej. JPG->WebP y
#    PNG->WebP pueden producir perdidas perceptuales distintas).
#    calidad se mantiene numerica (orden natural: 10 < 30 < 50 < 70 < 90)
# ------------------------------------------------------------------
df = pd.get_dummies(
    df, columns=["metodo", "formato_salida", "formato_entrada"], drop_first=False
)

# ------------------------------------------------------------------
# 6. LPIPS se conserva sin transformacion (decision ya justificada
#    en el analisis exploratorio)
# ------------------------------------------------------------------

# ------------------------------------------------------------------
# RESUMEN Y GUARDADO
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("RESUMEN DE LA LIMPIEZA")
print("=" * 70)
print(f"Filas iniciales : {n_inicial}")
print(f"Filas finales    : {len(df)}")
print(f"Columnas finales : {df.shape[1]}")
print(f"\nColumnas resultantes:\n{df.columns.tolist()}")

df.to_csv("csv/dataset_limpio.csv", index=False)
print("\nGuardado: csv/dataset_limpio.csv")

Dataset cargado: 29991 filas, 33 columnas

Filas eliminadas por lpips nulo: 2
Filas eliminadas por orig_entropia_espectral nulo: 2
Valores de orig_hue_var imputados con la mediana (0.31594): 392
Se agrego la columna 'orig_hue_var_faltante' como indicador binario
Filas restantes: 29987

RESUMEN DE LA LIMPIEZA
Filas iniciales : 29991
Filas finales    : 29987
Columnas finales : 33

Columnas resultantes:
['calidad', 'orig_media_lum', 'orig_std', 'orig_michelson', 'orig_hf', 'orig_lf', 'orig_edges_s3', 'orig_grad_mean', 'orig_grad_std', 'orig_glcm_contrast', 'orig_glcm_hom', 'orig_glcm_energy', 'orig_glcm_corr', 'orig_entropia_shannon', 'orig_entropia_espectral', 'orig_entropia_gradiente', 'orig_entropia_glcm', 'orig_sat_mean', 'orig_sat_var', 'orig_hue_var', 'orig_megapixeles', 'orig_aspect_ratio', 'lpips', 'orig_hue_var_faltante', 'metodo_ffmpeg', 'metodo_imagemagick', 'metodo_pillow', 'formato_salida_jpg', 'formato_salida_png', 'formato_salida_webp', 'formato_entrada_jpg', 'formato_entra